In [1]:
import os
import sys
import pandas as pd
from pathlib import Path
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import gc

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'
geo_root = repo_root.parent.parent / 'Data' / 'Geospatial'
gus_root = Path(os.getcwd()).parent.parent.parent.parent / "Data" / "GUS"

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))

# import local toolkit (try normal import first, fall back to loading from file)
try:
	import inequality_analyzers as inqA
	import local_utility_functions as luf
except Exception:
	import importlib.util
	toolkit_path = repo_root / 'Code' / 'tools' / 'inequality_analyzers.py'
	if toolkit_path.exists():
		spec = importlib.util.spec_from_file_location("inequality_analyzers", str(toolkit_path))
		stk = importlib.util.module_from_spec(spec)
		spec.loader.exec_module(stk)
	else:
		raise


In [2]:
# Load df and df_units from previously saved CSVs
df = pd.read_csv(gus_root / "metadata" / "bdl_variables_level6.csv", encoding="utf-8")
df_units = pd.read_csv(gus_root / "metadata" / "bdl_units_meta.csv", encoding="utf-8")

In [3]:
prg_05 = geo_root / "geometry" / "PRG_jednostki_administracyjne_2005"

# Read Obszary.shp file

gdf_2005 = gpd.read_file(prg_05 / "Obszary.shp", encoding="utf-8")
gdf_2005 = gdf_2005.to_crs(epsg=2180)  # Convert to EPSG:2180
#gdf_2005.plot()

In [4]:
prg_17 = geo_root / "geometry" / "PRG_jednostki_administracyjne_2017"

# Read gminy.shp file with polish characters

gdf_2017 = gpd.read_file(prg_17 / "gminy.shp")
gdf_2017 = gdf_2017.to_crs(epsg=2180)  # Convert to EPSG:2180
#gdf_2017.plot()

In [5]:
# Merging gdf_2017 and df_units on "jpt_kod_je" from gdf_2017 and "id" from df_units
# Creating columns "unitId" and "unitName" in gdf_2017 and "jpt_kod_je" in df_units for merging
gdf_2017['unitId'] = np.nan
gdf_2017['unitName'] = np.nan
df_units['jpt_kod_je'] = np.nan
for index, row in df_units.iterrows():
    gdf_code_from_units_id = luf.nuts_code_to_teryt(str(row['id']))
    where = np.where(gdf_2017['jpt_kod_je'].values == gdf_code_from_units_id)[0]
    if len(where) > 0:
        gdf_2017.at[where[0], 'unitId'] = row['id']
        gdf_2017.at[where[0], 'unitName'] = row['name']
        df_units.at[index, 'jpt_kod_je'] = gdf_code_from_units_id
    

/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_9851/2389908868.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Bochnia' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  gdf_2017.at[where[0], 'unitName'] = row['name']
/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_9851/2389908868.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1201011' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df_units.at[index, 'jpt_kod_je'] = gdf_code_from_units_id


In [6]:
df_units

,id,name,level,hasDescription,parentId,kind,years_available,early_year,late_year,number_of_years,description,jpt_kod_je
0,0,POLSKA,0,False,NaN,NaN,[],NaN,NaN,0,NaN,NaN
1,10000000000,MAKROREGION POŁUDNIOWY,1,False,0.000000e+00,NaN,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",1995.0,2026.0,32,NaN,NaN
2,11200000000,MAŁOPOLSKIE,2,True,1.000000e+10,NaN,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",1995.0,2026.0,32,Zmiana granic województwa z dniem 01.01.2002 r...,NaN
3,11212001000,Powiat bocheński,5,False,1.121200e+10,1.0,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",1995.0,2026.0,32,NaN,NaN
4,11212001011,Bochnia,6,False,1.121200e+10,1.0,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",1995.0,2026.0,32,NaN,1201011
...,...,...,...,...,...,...,...,...,...,...,...,...
4599,71427338042,Radziejowice,6,False,7.142734e+10,2.0,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",1995.0,2026.0,32,NaN,1438042
4600,71427338052,Wiskitki,6,True,7.142734e+10,2.0,"[1995, 1996, 1997, 1998, 1999, 2000, 2001, 200...",1995.0,2020.0,26,Zmiana rodzaju gminy z wiejskiego na miejsko-w...,1438052
4601,71427338053,Wiskitki,6,True,7.142734e+10,3.0,"[2021, 2022, 2023, 2024, 2025, 2026]",2021.0,2026.0,6,Zmiana rodzaju gminy z wiejskiego na miejsko-w...,NaN
4602,71427338054,Wiskitki - miasto,6,True,7.142734e+10,4.0,"[2021, 2022, 2023, 2024, 2025, 2026]",2021.0,2026.0,6,Zmiana rodzaju gminy z wiejskiego na miejsko-w...,NaN


In [7]:
def encode_level(row, col_names=['WOJ', 'POW', 'GMI']):
    c0, c1, c2 = col_names
    woj = row[c0]
    pow_ = row[c1]
    gmi = row[c2]
    if woj != '00' and pow_ == '00' and gmi == '00':
        return 2  # Voivodeship
    elif woj != '00' and pow_ != '00' and gmi == '00':
        return 5  # County
    elif woj != '00' and pow_ != '00' and gmi != '00':
        return 6  # Municipality
    else:
        return np.nan  # Undefined level

def encode_kind(row, col_name='RODZ'):
    val = str(row[col_name])
    if val == '0':
        return np.nan  # Not applicable
    mapping = {
        '1': 'urban',
        '2': 'rural',
        '3': 'urban-rural',
        '4': 'town',
        '5': 'village',
        '8': 'Warsaw district',
        '9': 'del. or district of a city'
    }
    return mapping.get(val, 'unknown')


In [8]:
# Oldest TERYT: TERC_Urzedowy_1999-01-01
# Newest TERYT: TERC_Urzedowy_2023-12-31

terc_1999 = pd.read_csv(geo_root / "TERC_Urzedowy_1999-01-01" / "TERC_Urzedowy_1999-01-01.csv", encoding="utf-8", sep=";")
terc_2024 = pd.read_csv(geo_root / "TERC_Urzedowy_2024-01-01" / "TERC_Urzedowy_2024-01-01.csv", encoding="utf-8", sep=";")

# File with all changes in TERYT codes in xml format
terc_changes = pd.read_xml(geo_root / "TERC_Urzedowy_zmiany_1999-01-01_2024-01-01.xml", encoding="utf-8")

In [9]:
# Change the columns 0, 1, 2 to string type with leading zeros of total length 2, 2, 2 respectively
# for 1999
terc_1999['WOJ'] = terc_1999['WOJ'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_1999['POW'] = terc_1999['POW'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_1999['GMI'] = terc_1999['GMI'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_1999['RODZ'] = terc_1999['RODZ'].apply(lambda x: str(int(x)).zfill(1) if not pd.isna(x) else "0")
terc_1999['level'] = terc_1999.apply(encode_level, axis=1)
terc_1999['kind'] = terc_1999.apply(encode_kind, axis=1)

# for 2023
terc_2024['WOJ'] = terc_2024['WOJ'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_2024['POW'] = terc_2024['POW'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_2024['GMI'] = terc_2024['GMI'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_2024['RODZ'] = terc_2024['RODZ'].apply(lambda x: str(int(x)).zfill(1) if not pd.isna(x) else "0")
terc_2024['level'] = terc_2024.apply(encode_level, axis=1)
terc_2024['kind'] = terc_2024.apply(encode_kind, axis=1)

In [10]:
# Create full TERYT code by concatenating the columns
terc_2024['id'] = terc_2024['WOJ'] + terc_2024['POW'] + terc_2024['GMI'] + terc_2024['RODZ']
terc_1999['id'] = terc_1999['WOJ'] + terc_1999['POW'] + terc_1999['GMI'] + terc_1999['RODZ']

In [11]:
terc_changes['WojPrzed'] = terc_changes['WojPrzed'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_changes['PowPrzed'] = terc_changes['PowPrzed'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_changes['GmiPrzed'] = terc_changes['GmiPrzed'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_changes['RodzPrzed'] = terc_changes['RodzPrzed'].apply(lambda x: str(int(x)).zfill(1) if not pd.isna(x) else "0")
terc_changes['WojPo'] = terc_changes['WojPo'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_changes['PowPo'] = terc_changes['PowPo'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_changes['GmiPo'] = terc_changes['GmiPo'].apply(lambda x: str(int(x)).zfill(2) if not pd.isna(x) else "00")
terc_changes['RodzPo'] = terc_changes['RodzPo'].apply(lambda x: str(int(x)).zfill(1) if not pd.isna(x) else "0")
terc_changes['id_before'] = terc_changes['WojPrzed'] + terc_changes['PowPrzed'] + terc_changes['GmiPrzed'] + terc_changes['RodzPrzed']
terc_changes['id_after'] = terc_changes['WojPo'] + terc_changes['PowPo'] + terc_changes['GmiPo'] + terc_changes['RodzPo']

gc.collect()

621

In [12]:
warsaw = terc_1999.loc[terc_1999['id'] == '1431000']
warsaw['RODZ'] = '1'
warsaw['NAZWA'] = 'Warszawa'
warsaw['NAZWA_DOD'] = 'gmina miejska, miasto stołeczne'
warsaw['level'] = 6
warsaw['kind'] = 'urban'
warsaw['id'] = '1431001'

terc_1999 = pd.concat([terc_1999, warsaw], ignore_index=True)

/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_9851/969226254.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  warsaw['RODZ'] = '1'
/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_9851/969226254.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  warsaw['NAZWA'] = 'Warszawa'
/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_9851/969226254.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer

In [13]:
terc_changes
# Indexes 110 to 127 incl. and 129
warsaw_indexes = [110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 129]
warsaw = terc_changes[terc_changes.index.isin(warsaw_indexes)]
warsaw['id_after'] = '1465011'
warsaw['WojPo'] = '14'
warsaw['PowPo'] = '65'
warsaw['GmiPo'] = '01'
warsaw['RodzPo'] = '1'

print(len(warsaw))

terc_changes = pd.concat([terc_changes, warsaw], ignore_index=True)

19


/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_9851/3784733376.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  warsaw['id_after'] = '1465011'
/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_9851/3784733376.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  warsaw['WojPo'] = '14'
/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_9851/3784733376.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_

In [14]:
display(terc_changes.loc[109,:])
terc_changes.at[109,"WojPrzed"] = '14'
terc_changes.at[109,"PowPrzed"] = '31'
terc_changes.at[109,"GmiPrzed"] = '00'
terc_changes.at[109,"RodzPrzed"] = '1'
terc_changes.at[109,"id_before"] = '1431001'
terc_changes.at[109,"NazwaPrzed"] = terc_changes.at[109,"NazwaPo"]
terc_changes.at[109,"NazwaDodatkowaPrzed"] = terc_changes.at[109,"NazwaDodatkowaPo"]

TypKorekty                                                    D
WojPrzed                                                     00
PowPrzed                                                     00
GmiPrzed                                                     00
RodzPrzed                                                     0
NazwaPrzed                                                 None
NazwaDodatkowaPrzed                                        None
StanPrzed                                            2002-01-01
WojPo                                                        14
PowPo                                                        65
GmiPo                                                        01
RodzPo                                                        1
NazwaPo                                                Warszawa
NazwaDodatkowaPo                gmina miejska, miasto stołeczne
WyodrebnionoZIdentyfikatora1                                NaN
WyodrebnionoZIdentyfikatora2            

In [15]:
terc_changes

,TypKorekty,WojPrzed,PowPrzed,GmiPrzed,RodzPrzed,NazwaPrzed,NazwaDodatkowaPrzed,StanPrzed,WojPo,PowPo,...,NazwaDodatkowaPo,WyodrebnionoZIdentyfikatora1,WyodrebnionoZIdentyfikatora2,WyodrebnionoZIdentyfikatora3,WlaczonoDoIdentyfikatora1,WlaczonoDoIdentyfikatora2,WlaczonoDoIdentyfikatora3,StanPo,id_before,id_after
0,M,02,20,02,2,Prusice,gmina wiejska,1999-01-01,02,20,...,gmina miejsko-wiejska,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0220022,0220023
1,D,00,00,00,0,None,None,1999-01-01,02,20,...,miasto,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0000000,0220024
2,D,00,00,00,0,None,None,1999-01-01,02,20,...,obszar wiejski,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0000000,0220025
3,M,06,18,12,2,Tyszowce,gmina wiejska,1999-01-01,06,18,...,gmina miejsko-wiejska,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0618122,0618123
4,D,00,00,00,0,None,None,1999-01-01,06,18,...,miasto,NaN,NaN,NaN,NaN,NaN,NaN,2000-01-01,0000000,0618124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
580,M,14,31,17,1,Warszawa-Wilanów,gmina miejska,2002-01-01,14,65,...,dzielnica,NaN,NaN,NaN,NaN,NaN,NaN,2002-10-27,1431171,1465011
581,M,14,31,18,1,Warszawa-Włochy,gmina miejska,2002-01-01,14,65,...,dzielnica,NaN,NaN,NaN,NaN,NaN,NaN,2002-10-27,1431181,1465011
582,M,14,31,10,8,Warszawa-Wola,dzielnica,2002-01-01,14,65,...,None,NaN,NaN,NaN,NaN,NaN,NaN,2002-10-27,1431108,1465011
583,M,14,31,11,8,Warszawa-Żoliborz,dzielnica,2002-01-01,14,65,...,None,NaN,NaN,NaN,NaN,NaN,NaN,2002-10-27,1431118,1465011


## TERYT Harmonization

Run the harmonization function to create a unified TERYT mega DataFrame spanning 1999-2024.

In [16]:
# Reload the local_utility_functions module to get the updated harmonize_teryt function
import importlib
importlib.reload(luf)

# Run harmonization
mega_df = luf.harmonize_teryt(terc_1999, terc_2024, terc_changes)

Harmonizing TERYT codes from 1999 to 2024...
  Year 2000: applying 17 changes...
  Year 2001: applying 12 changes...
  Year 2002: applying 120 changes...
  Year 2003: applying 6 changes...
  Year 2004: applying 10 changes...
  Year 2005: applying 4 changes...
  Year 2006: applying 7 changes...
  Year 2007: applying 6 changes...
  Year 2008: applying 6 changes...
  Year 2009: applying 15 changes...
  Year 2010: applying 21 changes...
  Year 2011: applying 16 changes...
  Year 2012: applying 1 changes...
  Year 2013: applying 2 changes...
  Year 2014: applying 18 changes...
  Year 2015: applying 10 changes...
  Year 2016: applying 17 changes...
  Year 2017: applying 16 changes...
  Year 2018: applying 28 changes...
  Year 2019: applying 31 changes...
  Year 2020: applying 12 changes...
  Year 2021: applying 32 changes...
  Year 2022: applying 30 changes...
  Year 2023: applying 45 changes...
  Year 2024: applying 103 changes...

Harmonization complete!
Total rows in mega DataFrame: 10737

In [17]:
# Validate: check that 2024 division matches the original terc_2024
division_2024 = mega_df[mega_df['year'] == 2024]
terc_ids = set(terc_2024['id'].unique())
mega_ids = set(division_2024['id'].unique())

print(f"Units in mega_df for 2024: {len(division_2024)}")
print(f"Units in terc_2024: {len(terc_2024)}")
print(f"Match: {len(division_2024) == len(terc_2024)}")

Units in mega_df for 2024: 4335
Units in terc_2024: 4332
Match: False


## Verification: Historical TERYT Codes

The `mega_df` now includes two new columns:
- `historical_codes`: list of all TERYT codes this unit has ever had
- `code_by_year`: dictionary mapping years to TERYT codes

Let's verify that historical code tracking works correctly using terc_changes.

In [18]:
# Add year_after column to terc_changes for verification
terc_changes['year_after'] = pd.to_datetime(terc_changes['StanPo']).dt.year

# Find units that had code changes from terc_changes
code_changes = terc_changes[(terc_changes['TypKorekty'] == 'M') & 
                            (terc_changes['id_before'] != '0000000') & 
                            (terc_changes['id_after'] != '0000000') &
                            (terc_changes['id_before'] != terc_changes['id_after'])]
print(f"Total code changes (M-type with different before/after IDs): {len(code_changes)}")

# Sample one change to verify
if len(code_changes) > 0:
    sample_change = code_changes.iloc[0]
    print(f"\nSample change: {sample_change['NazwaPrzed']} -> {sample_change['NazwaPo']}")
    print(f"  ID before: {sample_change['id_before']}")
    print(f"  ID after: {sample_change['id_after']}")
    print(f"  Year: {sample_change['year_after']}")

Total code changes (M-type with different before/after IDs): 243

Sample change: Prusice -> None
  ID before: 0220022
  ID after: 0220023
  Year: 2000


In [19]:
# Verify historical codes for a unit that changed
# Find the unit in mega_df by the "after" ID (in the final year)
if len(code_changes) > 0:
    sample_id_after = sample_change['id_after']
    sample_id_before = sample_change['id_before']
    sample_year = int(sample_change['year_after'])
    
    # Get the unit from 2024 (most recent)
    unit_record = mega_df[(mega_df['year'] == 2024) & (mega_df['id'] == sample_id_after)]
    
    if len(unit_record) > 0:
        hist_codes = unit_record.iloc[0]['historical_codes']
        cby = unit_record.iloc[0]['code_by_year']
        
        print(f"Unit: {unit_record.iloc[0]['NAZWA']} (current ID: {sample_id_after})")
        print(f"\nhistorical_codes: {hist_codes}")
        print(f"\ncode_by_year (years around the change):")
        for yr in range(sample_year - 2, sample_year + 3):
            if yr in cby:
                print(f"  {yr}: {cby[yr]}")
        
        # Verify the before code is in historical_codes
        print(f"\n✓ Old code '{sample_id_before}' in historical_codes: {sample_id_before in hist_codes}")
    else:
        print(f"Unit with ID {sample_id_after} not found in 2024")

Unit: Prusice (current ID: 0220023)

historical_codes: ['0220022', '0220023']

code_by_year (years around the change):
  1999: 0220022
  2000: 0220023
  2001: 0220023
  2002: 0220023

✓ Old code '0220022' in historical_codes: True


In [20]:
# Show units with multiple historical codes (units that had code changes)
units_2024 = mega_df[mega_df['year'] == 2024].copy()
units_2024['n_hist_codes'] = units_2024['historical_codes'].apply(lambda x: len(set(x)) if isinstance(x, list) else 0)
multi_code_units = units_2024[units_2024['n_hist_codes'] > 1]

print(f"Units with multiple historical codes: {len(multi_code_units)}")
print("\nSample of units with multiple historical codes:")
display(multi_code_units[['NAZWA', 'id', 'historical_codes', 'n_hist_codes']].head(10))

gc.collect()

Units with multiple historical codes: 219

Sample of units with multiple historical codes:


,NAZWA,id,historical_codes,n_hist_codes
103050,Pieszyce,0202033,"[0202031, 0202033]",2
103144,Olszyna,0210053,"[0210052, 0210053]",2
103226,Miękinia,0218033,"[0218032, 0218033]",2
103250,Prusice,0220023,"[0220022, 0220023]",2
103301,Kamieniec Ząbkowicki,0224033,"[0224032, 0224033]",2
103339,Wałbrzych,0265011,"[0263011, 0221091, 0265011]",3
103434,Bobrowniki,0408023,"[0408022, 0408023]",2
103439,Kikół,0408053,"[0408052, 0408053]",2
103507,Pruszcz,0414083,"[0414082, 0414083]",2
103566,Gąsawa,0419023,"[0419022, 0419023]",2


40

## Proof: mega_df contains all changes from terc_changes

We verify that every change recorded in `terc_changes` is reflected in the `mega_df`.

In [21]:
# Proof 1: Count changes by year in terc_changes vs mega_df
print("=== Verification: Changes in terc_changes vs mega_df ===\n")

# Changes by year from terc_changes
changes_by_year_terc = terc_changes.groupby('year_after').size().sort_index()

# Changes by year from mega_df (using when_changed column)
changed_records = mega_df[mega_df['if_changed'] == True]
# Get unique units changed per year (not duplicated across years)
changes_by_year_mega = changed_records.groupby('when_changed').apply(
    lambda x: x.drop_duplicates(subset=['id'])['id'].count()
).sort_index()

print("Changes per year:")
print(f"{'Year':<8} {'terc_changes':<15} {'mega_df':<15}")
print("-" * 40)
for year in range(2000, 2025):
    terc_count = changes_by_year_terc.get(year, 0)
    mega_count = changes_by_year_mega.get(float(year), 0)
    match = "✓" if terc_count == mega_count else "≈"
    print(f"{year:<8} {terc_count:<15} {mega_count:<15} {match}")

print(f"\nTotal changes in terc_changes: {len(terc_changes)}")
print(f"Total changed records in mega_df: {changed_records['id'].nunique()} unique units")

=== Verification: Changes in terc_changes vs mega_df ===

Changes per year:
Year     terc_changes    mega_df        
----------------------------------------
2000     17              17              ✓
2001     12              12              ✓
2002     120             96              ≈
2003     6               5               ≈
2004     10              10              ✓
2005     4               4               ✓
2006     7               7               ✓
2007     6               6               ✓
2008     6               6               ✓
2009     15              15              ✓
2010     21              21              ✓
2011     16              16              ✓
2012     1               1               ✓
2013     2               2               ✓
2014     18              18              ✓
2015     10              9               ≈
2016     17              17              ✓
2017     16              16              ✓
2018     28              25              ≈
2019     31              

/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_9851/3933702684.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  changes_by_year_mega = changed_records.groupby('when_changed').apply(


In [22]:
# Proof 2: For ID changes specifically, verify they are tracked in historical_codes
print("=== Verification: ID changes tracked in historical_codes ===\n")

# Get all M-type changes where ID actually changed
id_changes = terc_changes[(terc_changes['TypKorekty'] == 'M') & 
                          (terc_changes['id_before'] != '0000000') & 
                          (terc_changes['id_after'] != '0000000') &
                          (terc_changes['id_before'] != terc_changes['id_after'])]

print(f"Total ID changes in terc_changes: {len(id_changes)}")

# Sample verification for multiple ID changes
verified_count = 0
failed_count = 0
failed_examples = []

for idx, row in id_changes.head(50).iterrows():  # Check first 50
    id_after = row['id_after']
    id_before = row['id_before']
    
    # Find this unit in 2024
    unit_2024 = mega_df[(mega_df['year'] == 2024) & (mega_df['id'] == id_after)]
    
    if len(unit_2024) > 0:
        hist_codes = unit_2024.iloc[0]['historical_codes']
        if isinstance(hist_codes, list) and id_before in hist_codes:
            verified_count += 1
        else:
            failed_count += 1
            failed_examples.append((id_before, id_after, hist_codes))
    else:
        # Unit might have been removed or merged
        pass

print(f"Verified: {verified_count}/{verified_count + failed_count}")
if failed_examples:
    print(f"Failed examples: {failed_examples[:3]}")

=== Verification: ID changes tracked in historical_codes ===

Total ID changes in terc_changes: 243
Verified: 47/47


## Save Final DataFrame

In [23]:
# Save the harmonized TERYT dataframe
mega_df.to_csv(geo_root / "teryt_df.csv", index=False, encoding="utf-8")
print(f"Saved mega_df to {geo_root / 'teryt_df.csv'}")
print(f"Total rows: {len(mega_df)}, Columns: {list(mega_df.columns)}")

gc.collect()

Saved mega_df to /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/Geospatial/teryt_df.csv
Total rows: 107372, Columns: ['year', 'WOJ', 'POW', 'GMI', 'RODZ', 'id', 'NAZWA', 'NAZWA_DOD', 'level', 'kind', 'STAN_NA', 'if_changed', 'when_changed', 'notes', 'historical_codes', 'code_by_year']


0

In [ ]:
mega_df

,year,WOJ,POW,GMI,RODZ,id,NAZWA,NAZWA_DOD,level,kind,STAN_NA,if_changed,when_changed,notes,historical_codes,code_by_year
0,1999,02,00,00,0,0200000,DOLNOŚLĄSKIE,województwo,2,NaN,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}",[0200000],"{1999: '0200000', 2000: '0200000', 2001: '0200..."
1,1999,02,01,00,0,0201000,bolesławiecki,powiat,5,NaN,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}",[0201000],"{1999: '0201000', 2000: '0201000', 2001: '0201..."
2,1999,02,01,01,1,0201011,Bolesławiec,gmina miejska,6,urban,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}",[0201011],"{1999: '0201011', 2000: '0201011', 2001: '0201..."
3,1999,02,01,02,2,0201022,Bolesławiec,gmina wiejska,6,rural,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}",[0201022],"{1999: '0201022', 2000: '0201022', 2001: '0201..."
4,1999,02,01,03,2,0201032,Gromadka,gmina wiejska,6,rural,1999-01-01,False,NaN,"{'number_of_changes': 0, 'changes': []}",[0201032],"{1999: '0201032', 2000: '0201032', 2001: '0201..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
107367,2024,26,12,01,5,2612015,Bogoria,obszar wiejski,6,village,2024-01-01,True,2024.0,"{'number_of_changes': 1, 'changes': ['NEW: gmi...",[2612015],{2024: '2612015'}
107368,2024,30,08,06,4,3008064,Rychtal,miasto,6,town,2024-01-01,True,2024.0,"{'number_of_changes': 1, 'changes': ['NEW: gmi...",[3008064],{2024: '3008064'}
107369,2024,30,08,06,5,3008065,Rychtal,obszar wiejski,6,village,2024-01-01,True,2024.0,"{'number_of_changes': 1, 'changes': ['NEW: gmi...",[3008065],{2024: '3008065'}
107370,2024,30,28,04,4,3028044,Mieścisko,miasto,6,town,2024-01-01,True,2024.0,"{'number_of_changes': 1, 'changes': ['NEW: gmi...",[3028044],{2024: '3028044'}
